# Multi-Label Emotion Classification on GoEmotions
### Fine-tuning an encoder-only Transformer (IE624 · Generative and Agentic AI)

**What this notebook does, in one paragraph.** We fine-tune a pretrained RoBERTa encoder to read a short Reddit comment and decide which of 28 labels (27 emotions + *neutral*) apply. Because a comment can carry several emotions at once, and because nobody tells the model *how many* labels to return, we treat each label as its own yes/no question (sigmoid outputs, binary cross-entropy), and then learn a *decision rule* on the validation set: one probability threshold per label, plus a fallback for the rare case where no label fires.

**Rules we respect** (so a reader can check us):
- Training uses **only** the official training split. The validation split is used for choosing the checkpoint and tuning the decision rule. The test split is never touched until a single, clearly marked cell at the very end.
- One model, no ensembles, no external data, no emotion-pretrained checkpoints, single GPU (<= 16 GB).
- The encoder itself is fine-tuned (full fine-tuning), not just a head on frozen features.

**How the notebook is organised**

| Part | What happens | Needs GPU? |
|---|---|---|
| 0 | Setup, reproducibility info, configuration | - |
| 1 | Data: loading, inspection, tokenisation | - |
| 2 | Model and parameter counts | yes |
| 3 | Training (guarded by `RUN_TRAINING`) | yes |
| 4 | Decision rule: thresholds + empty-prediction fallback, tuned on validation | no |
| 5 | Saving and zipping the checkpoint | no |
| 6 | **Inference section** (self-contained; can be run in a fresh session) | optional |
| 7 | One-time test evaluation (off by default) | optional |

**Kaggle settings to use:** Accelerator = *GPU T4 x2* (we deliberately use only one of them, see below), Internet = *On* (to download the model and dataset), Persistence = *Files only* is helpful.

**Workflow:** run once with `MODE = "smoke"` (a quick end-to-end check with `roberta-base`), then switch to `MODE = "full"` (the real `roberta-large` run).

## Part 0 · Setup and reproducibility

We pin the notebook to **one** GPU by setting `CUDA_VISIBLE_DEVICES` *before* PyTorch is imported. Kaggle's "T4 x2" machine would otherwise expose two GPUs, and the assignment allows only one.

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"        # single GPU only (assignment rule)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import sys, json, time, random, math, platform, shutil, zipfile, glob
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import transformers
import datasets
import scipy
from scipy.ndimage import uniform_filter1d
from tqdm.auto import tqdm
from datasets import load_dataset
from transformers import (AutoConfig, AutoTokenizer, AutoModelForSequenceClassification,
                          get_linear_schedule_with_warmup)

assert torch.cuda.is_available(), "Turn on a GPU accelerator in the Kaggle sidebar."
assert torch.cuda.device_count() == 1, "More than one GPU visible; the task allows exactly one."
DEVICE = torch.device("cuda")

VERSIONS = {
    "python": platform.python_version(),
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "datasets": datasets.__version__,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scipy": scipy.__version__,
    "cuda": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0),
    "gpu_memory_GB": round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1),
}
for k, v in VERSIONS.items():
    print(f"{k:>14}: {v}")

        python: 3.13.15
         torch: 2.11.0+cu128
  transformers: 5.16.1
      datasets: 4.8.5
         numpy: 2.1.3
        pandas: 2.3.3
         scipy: 1.16.3
          cuda: 12.8
           gpu: Tesla T4
 gpu_memory_GB: 15.6


### Configuration

All the knobs live in one place. Two presets:

- **`smoke`**: `roberta-base`, 2 epochs. Its only job is to prove the *whole pipeline* (train, tune, save, reload, predict) works before we spend the big GPU budget.
- **`full`**: `roberta-large`, 5 epochs. This is the model we intend to submit.

**Why RoBERTa-large?** It is a strong encoder-only model that is known to do well on GoEmotions-style tasks. At 355M parameters it fits comfortably in 16 GB when we use mixed precision and short sequences (see Part 1 for why 64 tokens is plenty). We use **full fine-tuning** (all weights updated), which at this size needs roughly 6-9 GB, so we don't need adapters.

**Why these hyper-parameters?** Large Transformers are touchy about learning rate: too high and training can collapse into predicting "nothing" for every label. So we use a modest peak learning rate (2e-5), a 6% warm-up, linear decay, and **layer-wise learning-rate decay**: layers close to the input (which hold general language knowledge) change more slowly than layers close to the output, and the freshly-initialised classification head learns fastest.

In [2]:
MODE = "smoke"          # <- "smoke" first, then change to "full"
RUN_TRAINING = True     # set False to skip training (e.g. after a restart, to redo Parts 4-5 from saved files)
SEED = 42

PRESETS = {
    "smoke": dict(model_name="FacebookAI/roberta-base",  epochs=2, lr=4e-5, llrd=0.90, batch_size=32),
    "full":  dict(model_name="FacebookAI/roberta-large", epochs=5, lr=2e-5, llrd=0.95, batch_size=32),
}

CFG = dict(PRESETS[MODE])
CFG.update(
    mode=MODE, seed=SEED,
    dataset_name="google-research-datasets/go_emotions", dataset_config="simplified",
    max_len=64,               # justified in Part 1
    warmup_ratio=0.06, weight_decay=0.01, grad_clip=1.0,
    head_lr_mult=3.0,         # classification head learns 3x faster than the top encoder layer
    pos_weight_power=0.0,     # 0 = plain BCE. >0 enables mild positive-class re-weighting (see Part 3)
    patience=2,               # stop if the validation score has not improved for this many epochs
    fp16=True,
)

WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./work")
WORK.mkdir(parents=True, exist_ok=True)
OUT_DIR = WORK / f"goemotions_checkpoint_{MODE}"      # everything we need to ship lives here
OUT_DIR.mkdir(parents=True, exist_ok=True)

def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
set_seed(SEED)

# Resolve the exact commit (revision) of the base model and the dataset so the run is reproducible.
from huggingface_hub import HfApi
try:
    api = HfApi()
    CFG["model_revision"]   = api.model_info(CFG["model_name"]).sha
    CFG["dataset_revision"] = api.dataset_info(CFG["dataset_name"]).sha
except Exception as e:
    print("Could not resolve revisions automatically:", e)
    CFG["model_revision"], CFG["dataset_revision"] = None, None

print(json.dumps(CFG, indent=2))

{
  "model_name": "FacebookAI/roberta-base",
  "epochs": 2,
  "lr": 4e-05,
  "llrd": 0.9,
  "batch_size": 32,
  "mode": "smoke",
  "seed": 42,
  "dataset_name": "google-research-datasets/go_emotions",
  "dataset_config": "simplified",
  "max_len": 64,
  "warmup_ratio": 0.06,
  "weight_decay": 0.01,
  "grad_clip": 1.0,
  "head_lr_mult": 3.0,
  "pos_weight_power": 0.0,
  "patience": 2,
  "fp16": true,
  "model_revision": "e2da8e2f811d1448a5b465c236feacd80ffbac7b",
  "dataset_revision": "add492243ff905527e67aeb8b80c082af02207c3"
}


## Part 1 · Data

We use the **simplified** GoEmotions configuration with its **official splits**, untouched. Each example is a comment plus a *list* of integer label IDs (1 or more). We keep the original ID-to-name mapping, and we keep examples where *neutral* co-occurs with other labels exactly as annotated (no cleaning, no de-duplication).

The test split gets loaded together with the others (that is how the library works), but **no cell touches it until Part 7**.

In [3]:
ds = load_dataset(CFG["dataset_name"], CFG["dataset_config"], revision=CFG["dataset_revision"])
label_names = ds["train"].features["labels"].feature.names
NUM_LABELS = len(label_names)
NEUTRAL_ID = label_names.index("neutral")

assert NUM_LABELS == 28 and NEUTRAL_ID == 27
assert (len(ds["train"]), len(ds["validation"]), len(ds["test"])) == (43410, 5426, 5427)
print({k: len(v) for k, v in ds.items()})
print(label_names)
ds["train"][0], ds["train"][1]

README.md:   0%|          | 0.00/9.40k [00:00<?, ?B/s]

simplified/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.77MB            

simplified/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

simplified/validation-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B /  350kB            

simplified/validation-00000-of-00001.par(…): downloading bytes:           |  0.00B            

simplified/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  347kB            

... (2 outputs left out)

Generating validation split:   0%|          | 0/5426 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5427 [00:00<?, ? examples/s]

{'train': 43410, 'validation': 5426, 'test': 5427}
['admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring', 'confusion', 'curiosity', 'desire', 'disappointment', 'disapproval', 'disgust', 'embarrassment', 'excitement', 'fear', 'gratitude', 'grief', 'joy', 'love', 'nervousness', 'optimism', 'pride', 'realization', 'relief', 'remorse', 'sadness', 'surprise', 'neutral']


({'text': "My favourite food is anything I didn't have to cook myself.",
  'labels': [27],
  'id': 'eebbqej'},
 {'text': 'Now if he does off himself, everyone will think hes having a laugh screwing with people instead of actually dead',
  'labels': [27],
  'id': 'ed00q6i'})

In [4]:
def to_multihot(label_lists, n=NUM_LABELS):
    Y = np.zeros((len(label_lists), n), dtype=np.float32)
    for i, labs in enumerate(label_lists):
        Y[i, labs] = 1.0
    return Y

train_texts, val_texts = ds["train"]["text"], ds["validation"]["text"]
Y_train = to_multihot(ds["train"]["labels"])
Y_val   = to_multihot(ds["validation"]["labels"])

stats = pd.DataFrame({
    "label": label_names,
    "train_n": Y_train.sum(0).astype(int),
    "val_n":   Y_val.sum(0).astype(int),
})
stats["train_%"] = (100 * stats["train_n"] / len(Y_train)).round(2)
print(f"Average labels per example (train): {Y_train.sum(1).mean():.3f}")
print(f"Share of examples with 2+ labels:   {(Y_train.sum(1) > 1).mean():.1%}")
print(f"'neutral' together with another label: {((Y_train[:, NEUTRAL_ID] == 1) & (Y_train.sum(1) > 1)).sum()} train examples")
stats.sort_values("train_n", ascending=False).reset_index(drop=True)

Average labels per example (train): 1.177
Share of examples with 2+ labels:   16.4%
'neutral' together with another label: 1396 train examples


,label,train_n,val_n,train_%
0,neutral,14219,1766,32.76
1,admiration,4130,488,9.51
2,approval,2939,397,6.77
3,gratitude,2662,358,6.13
4,annoyance,2470,303,5.69
5,amusement,2328,303,5.36
6,curiosity,2191,248,5.05
7,love,2086,252,4.81
8,disapproval,2022,292,4.66
9,optimism,1581,209,3.64


**Reading the table above.** The label distribution is heavily skewed: *neutral*, *admiration* and *approval* are common, while *grief*, *pride*, *relief* and *nervousness* have well under 1% of examples. Since the final score is **Macro-F1** (every label counts equally), the rare labels matter just as much as the common ones. A single global threshold such as 0.5 would almost never fire for them, which is the main reason Part 4 tunes thresholds per label.

### Tokenisation and sequence length

In [5]:
tokenizer = AutoTokenizer.from_pretrained(CFG["model_name"], revision=CFG["model_revision"], use_fast=True)

lens = np.array([len(x) for x in tokenizer(train_texts, truncation=False)["input_ids"]])
print(f"mean {lens.mean():.1f} | median {np.median(lens):.0f} | 95th pct {np.percentile(lens, 95):.0f} | "
      f"99th pct {np.percentile(lens, 99):.0f} | max {lens.max()}")
print(f"Share of training comments that fit in {CFG['max_len']} tokens: {(lens <= CFG['max_len']).mean():.2%}")

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

ValueError: text input must be of type `str` (single example), `list[str]` (batch or single pretokenized example) or `list[list[str]]` (batch of pretokenized examples) or `list[tuple[list[str], list[str]]]` (batch of pretokenized sequence pairs).

GoEmotions comments are short (Reddit comments were capped in length by the dataset authors). Almost everything fits in 64 tokens, so truncating at 64 loses essentially nothing while keeping memory and compute low. We also use **dynamic padding**: each batch is padded only to its own longest sequence, not to 64, which saves a lot of GPU time since the typical comment is around 15 tokens.

In [ ]:
def encode(texts):
    return tokenizer(list(texts), truncation=True, max_length=CFG["max_len"])["input_ids"]

enc_train, enc_val = encode(train_texts), encode(val_texts)

class TextDataset(torch.utils.data.Dataset):
    def __init__(self, ids, Y): self.ids, self.Y = ids, Y
    def __len__(self): return len(self.ids)
    def __getitem__(self, i): return {"input_ids": self.ids[i], "labels": self.Y[i]}

def collate(batch):
    padded = tokenizer.pad({"input_ids": [b["input_ids"] for b in batch]}, return_tensors="pt")
    padded["labels"] = torch.tensor(np.stack([b["labels"] for b in batch]))
    return padded

g = torch.Generator(); g.manual_seed(SEED)
train_loader = torch.utils.data.DataLoader(TextDataset(enc_train, Y_train), batch_size=CFG["batch_size"],
                                           shuffle=True, collate_fn=collate, generator=g, num_workers=2)
print("train batches per epoch:", len(train_loader))

## Part 2 · Model

We use Hugging Face's `AutoModelForSequenceClassification` configured with `problem_type="multi_label_classification"` and 28 outputs. For RoBERTa this means:

1. the encoder reads the comment,
2. the final hidden state of the first token (`<s>`, RoBERTa's summary token) goes through a small head (dense layer, tanh, dropout, linear layer),
3. the head outputs **28 logits**, one per label. Each logit is turned into a probability with a **sigmoid** (not a softmax), so the labels are not forced to compete. Several can be "on" at once.

Using the library's standard class (instead of a hand-made head) also means the saved checkpoint reloads with one line, with no custom code, which keeps the submission easy to verify.

In [ ]:
def build_model():
    config = AutoConfig.from_pretrained(
        CFG["model_name"], revision=CFG["model_revision"],
        num_labels=NUM_LABELS,
        id2label={i: n for i, n in enumerate(label_names)},
        label2id={n: i for i, n in enumerate(label_names)},
        problem_type="multi_label_classification",
    )
    return AutoModelForSequenceClassification.from_pretrained(
        CFG["model_name"], revision=CFG["model_revision"], config=config)

def count_params(m):
    total = sum(p.numel() for p in m.parameters())
    trainable = sum(p.numel() for p in m.parameters() if p.requires_grad)
    head = sum(p.numel() for n, p in m.named_parameters() if n.startswith("classifier"))
    return dict(total=total, trainable=trainable, classification_head=head, encoder=total - head)

model = build_model().to(DEVICE)
PARAM_COUNTS = count_params(model)
print(json.dumps(PARAM_COUNTS, indent=2))
print(model.classifier)

## Metrics and decision-rule toolkit

Before training, we define the helpers we will use both to monitor training and to build the decision rule. They are plain NumPy so they run instantly.

- `macro_f1`: exactly the formula in the assignment: per-label F1 = 2TP / (2TP + FP + FN), with F1 = 0 when the denominator is zero, averaged over **all 28 labels**.
- `f1_curves`: for every label, F1 as a function of the threshold (grid from 0.05 to 0.80).
- `fit_thresholds`: three ways to choose thresholds (explained in Part 4).
- `apply_rule`: turns probabilities into label sets, including the fallback for empty predictions.

In [ ]:
GRID = np.round(np.arange(0.05, 0.81, 0.01), 2)

def f1_per_label(Y, P):
    Y, P = Y.astype(bool), P.astype(bool)
    tp = (Y & P).sum(0); fp = (~Y & P).sum(0); fn = (Y & ~P).sum(0)
    denom = 2 * tp + fp + fn
    return np.where(denom > 0, 2 * tp / np.maximum(denom, 1), 0.0)

def macro_f1(Y, P):
    return float(f1_per_label(Y, P).mean())

def micro_f1(Y, P):
    Y, P = Y.astype(bool), P.astype(bool)
    tp = (Y & P).sum(); fp = (~Y & P).sum(); fn = (Y & ~P).sum()
    return float(2 * tp / max(2 * tp + fp + fn, 1))

def f1_curves(Y, probs, grid=GRID):
    # returns an (n_labels, n_thresholds) matrix of F1 values
    curves = np.zeros((Y.shape[1], len(grid)))
    for j in range(Y.shape[1]):
        pred = probs[:, j:j + 1] >= grid[None, :]
        y = Y[:, j:j + 1].astype(bool)
        tp = (pred & y).sum(0); fp = (pred & ~y).sum(0); fn = (~pred & y).sum(0)
        d = 2 * tp + fp + fn
        curves[j] = np.where(d > 0, 2 * tp / np.maximum(d, 1), 0.0)
    return curves

def _middle_argmax(curve):
    # if several thresholds tie for the best F1, take the middle one (more stable than the first)
    best = np.flatnonzero(curve >= curve.max() - 1e-12)
    return best[len(best) // 2]

def fit_thresholds(Y, probs, strategy="smooth_shrunk", smooth=5, k=30):
    # strategy: "global" | "raw" | "smooth_shrunk"
    curves = f1_curves(Y, probs)
    t_global = GRID[_middle_argmax(curves.mean(0))]
    if strategy == "global":
        return np.full(Y.shape[1], t_global)
    if strategy == "raw":
        return np.array([GRID[_middle_argmax(c)] for c in curves])
    # smooth_shrunk: (1) smooth each label's F1-vs-threshold curve so one lucky spike does not win,
    # (2) pull the threshold toward the global one in proportion to how few positives the label has.
    sm = uniform_filter1d(curves, size=smooth, axis=1, mode="nearest")
    t_label = np.array([GRID[_middle_argmax(c)] for c in sm])
    n_pos = Y.sum(0)
    w = n_pos / (n_pos + k)
    return w * t_label + (1 - w) * t_global

def apply_rule(probs, thr, fallback="ratio", neutral_id=NEUTRAL_ID):
    # probs: (N, L). thr: (L,). Returns a boolean matrix with at least one True per row.
    pred = probs >= thr[None, :]
    empty = np.flatnonzero(~pred.any(1))
    if len(empty):
        if fallback == "ratio":      # label whose probability is furthest above its own threshold
            idx = (probs[empty] / thr[None, :]).argmax(1)
        elif fallback == "prob":     # simply the most probable label
            idx = probs[empty].argmax(1)
        elif fallback == "neutral":
            idx = np.full(len(empty), neutral_id)
        else:
            raise ValueError(fallback)
        pred[empty, idx] = True
    return pred

# quick self-test on synthetic data so we know the maths is right
_rng = np.random.default_rng(0)
_Y = (_rng.random((500, 4)) < [0.5, 0.2, 0.05, 0.01]).astype(np.float32)
_P = np.clip(_Y * 0.6 + _rng.random((500, 4)) * 0.4, 0, 1)
assert apply_rule(_P, np.full(4, 0.99)).any(1).all()            # fallback guarantees >= 1 label
assert abs(macro_f1(_Y, _Y) - np.mean(_Y.sum(0) > 0)) < 1e-9     # perfect prediction; empty labels score 0
print("toolkit self-test passed")

## Part 3 · Training

A short explanation of each piece, since these are the choices a reader is most likely to ask about:

- **Loss: binary cross-entropy on logits** (`BCEWithLogitsLoss`), one binary problem per label. This is the standard, correct objective for multi-label classification. An optional `pos_weight` can boost rare labels (`pos_weight_power > 0`), but it is off by default, because threshold tuning in Part 4 already compensates for imbalance and re-weighting can make probabilities less well-behaved.
- **Mixed precision (fp16)**: roughly halves memory and speeds up the T4. We keep the loss computation in fp32 for stability.
- **AdamW + warm-up + linear decay + gradient clipping**: the standard, stable recipe for fine-tuning Transformers.
- **Layer-wise LR decay**: described in Part 0.
- **Checkpoint selection**: after every epoch we score the validation set and keep the best epoch. The selection score is the Macro-F1 under the *best single global threshold*. We deliberately use this simple one-parameter score for picking epochs (instead of the fully tuned per-label rule) because it is less prone to overfitting the validation set. The more detailed rule is tuned afterwards, once the checkpoint is fixed.
- **Evaluation speed**: validation batches are sorted by length so padding is minimal.

In [ ]:
def build_optimizer(model, lr, llrd, head_mult, wd):
    no_decay = ("bias", "LayerNorm.weight", "LayerNorm.bias")
    n_layers = model.config.num_hidden_layers
    named = list(model.named_parameters())
    used, groups = set(), []

    def add(items, lr_):
        for n, _ in items: used.add(n)
        dec   = [p for n, p in items if not any(nd in n for nd in no_decay)]
        nodec = [p for n, p in items if any(nd in n for nd in no_decay)]
        if dec:   groups.append({"params": dec,   "lr": lr_, "weight_decay": wd})
        if nodec: groups.append({"params": nodec, "lr": lr_, "weight_decay": 0.0})

    add([(n, p) for n, p in named if n.startswith("classifier")], lr * head_mult)
    for i in range(n_layers):
        add([(n, p) for n, p in named if f"encoder.layer.{i}." in n], lr * (llrd ** (n_layers - 1 - i)))
    add([(n, p) for n, p in named if "embeddings" in n], lr * (llrd ** n_layers))
    leftovers = [(n, p) for n, p in named if n not in used]
    if leftovers: add(leftovers, lr)
    assert sum(p.numel() for g_ in groups for p in g_["params"]) == sum(p.numel() for _, p in named)
    return torch.optim.AdamW(groups, betas=(0.9, 0.999), eps=1e-8)

@torch.inference_mode()
def predict_probs(model, id_lists, batch_size=128):
    model.eval()
    order = np.argsort([len(x) for x in id_lists])
    out = np.zeros((len(id_lists), NUM_LABELS), dtype=np.float32)
    for s in range(0, len(order), batch_size):
        idx = order[s:s + batch_size]
        batch = tokenizer.pad({"input_ids": [id_lists[i] for i in idx]}, return_tensors="pt").to(DEVICE)
        with torch.autocast("cuda", dtype=torch.float16, enabled=CFG["fp16"]):
            logits = model(**batch).logits
        out[idx] = torch.sigmoid(logits.float()).cpu().numpy()
    return out

def quick_val_scores(Y, probs):
    curves = f1_curves(Y, probs)
    g_idx = _middle_argmax(curves.mean(0))
    return {
        "macroF1@0.5": macro_f1(Y, probs >= 0.5),
        "macroF1_best_global": float(curves.mean(0)[g_idx]),
        "best_global_threshold": float(GRID[g_idx]),
    }

In [ ]:
history, best = [], {"score": -1.0, "epoch": None}

if RUN_TRAINING:
    set_seed(SEED)
    model = build_model().to(DEVICE)
    optimizer = build_optimizer(model, CFG["lr"], CFG["llrd"], CFG["head_lr_mult"], CFG["weight_decay"])
    total_steps = len(train_loader) * CFG["epochs"]
    scheduler = get_linear_schedule_with_warmup(optimizer, int(CFG["warmup_ratio"] * total_steps), total_steps)
    scaler = torch.amp.GradScaler("cuda", enabled=CFG["fp16"])

    pos_weight = None
    if CFG["pos_weight_power"] > 0:
        pos = Y_train.sum(0); pos_weight = torch.tensor(((len(Y_train) - pos) / pos) ** CFG["pos_weight_power"],
                                                        dtype=torch.float32, device=DEVICE)
    loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    bad_epochs = 0
    for epoch in range(1, CFG["epochs"] + 1):
        model.train(); t0 = time.time(); running = []
        pbar = tqdm(train_loader, desc=f"epoch {epoch}/{CFG['epochs']}")
        for batch in pbar:
            batch = {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items()}
            labels = batch.pop("labels")
            with torch.autocast("cuda", dtype=torch.float16, enabled=CFG["fp16"]):
                logits = model(**batch).logits
            loss = loss_fn(logits.float(), labels)          # loss in fp32 for stability
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), CFG["grad_clip"])
            scaler.step(optimizer); scaler.update(); scheduler.step()
            running.append(loss.item())
            if len(running) % 50 == 0: pbar.set_postfix(loss=f"{np.mean(running[-50:]):.4f}")

        val_probs = predict_probs(model, enc_val)
        scores = quick_val_scores(Y_val, val_probs)
        rec = {"epoch": epoch, "train_loss": float(np.mean(running)), "minutes": round((time.time() - t0) / 60, 1), **scores}
        history.append(rec); print(rec)

        if scores["macroF1_best_global"] > best["score"]:
            best = {"score": scores["macroF1_best_global"], "epoch": epoch}; bad_epochs = 0
            model.save_pretrained(OUT_DIR, safe_serialization=True)
            tokenizer.save_pretrained(OUT_DIR)
            np.save(OUT_DIR / "val_probs.npy", val_probs)          # lets us redo Part 4 without a GPU
            print(f"  -> new best (epoch {epoch}); checkpoint saved")
        else:
            bad_epochs += 1
            if bad_epochs >= CFG["patience"]:
                print("  early stopping"); break

    json.dump(history, open(OUT_DIR / "training_log.json", "w"), indent=2)
    print("best epoch:", best)
    del optimizer, scheduler, scaler; torch.cuda.empty_cache()
else:
    print("RUN_TRAINING=False: skipping training; expecting a saved checkpoint in", OUT_DIR)

In [ ]:
# Training curves (only if we trained in this session)
if history:
    import matplotlib.pyplot as plt
    h = pd.DataFrame(history)
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
    ax[0].plot(h.epoch, h.train_loss, marker="o"); ax[0].set_title("Training loss"); ax[0].set_xlabel("epoch")
    ax[1].plot(h.epoch, h["macroF1_best_global"], marker="o", label="best global threshold")
    ax[1].plot(h.epoch, h["macroF1@0.5"], marker="o", label="threshold 0.5")
    ax[1].set_title("Validation Macro-F1"); ax[1].set_xlabel("epoch"); ax[1].legend()
    plt.tight_layout(); plt.show()
    display(h)

## Part 4 · The decision rule (tuned on validation only)

The model gives 28 probabilities per comment. We still have to decide *which* labels to output and *how many*. The assignment says the count is not supplied, so we use a per-label threshold rule, and then deal with the leftover case where no probability crosses its threshold.

**Step A. Choose how to pick thresholds.** We compare three options:

1. **global**: one threshold shared by all labels.
2. **raw**: each label gets the threshold that maximises its own validation F1.
3. **smooth_shrunk**: like *raw*, but (a) each label's F1-vs-threshold curve is smoothed first, so a single lucky spike does not decide the threshold, and (b) the threshold is pulled toward the global one for labels with few positive examples (weight `n / (n + k)`). Labels like *grief* have only a handful of validation positives, so their "best" threshold is mostly noise; shrinkage is a classic way of handling that.

The catch: if we pick thresholds on the validation set and then score them on the same set, the numbers are over-optimistic, and "raw" would always look best. To compare strategies **fairly using validation data only**, we use *cross-fitting*: repeatedly split validation in half at random, fit thresholds on one half, and score on the other half. The strategy with the best held-out average wins.

**Step B. Choose the fallback** for inputs where nothing fires (three candidates: highest `probability / threshold` ratio, highest raw probability, or *neutral*).

In [ ]:
val_probs = np.load(OUT_DIR / "val_probs.npy")
assert val_probs.shape == Y_val.shape

def cross_fit(Y, probs, strategy, repeats=40, seed=0, **kw):
    rng = np.random.default_rng(seed); scores = []
    for _ in range(repeats):
        perm = rng.permutation(len(Y)); a, b = perm[: len(Y) // 2], perm[len(Y) // 2:]
        for tr, te in ((a, b), (b, a)):
            thr = fit_thresholds(Y[tr], probs[tr], strategy, **kw)
            scores.append(macro_f1(Y[te], apply_rule(probs[te], thr, "ratio")))
    return np.mean(scores), np.std(scores)

candidates = {
    "global":                  dict(strategy="global"),
    "raw":                     dict(strategy="raw"),
    "smooth_shrunk (k=10)":    dict(strategy="smooth_shrunk", smooth=5, k=10),
    "smooth_shrunk (k=30)":    dict(strategy="smooth_shrunk", smooth=5, k=30),
    "smooth_shrunk (k=100)":   dict(strategy="smooth_shrunk", smooth=5, k=100),
}
rows = []
for name, kw in candidates.items():
    m, s = cross_fit(Y_val, val_probs, **kw)
    rows.append({"strategy": name, "held-out macro-F1 (mean)": round(m, 4), "std": round(s, 4)})
cmp_df = pd.DataFrame(rows).sort_values("held-out macro-F1 (mean)", ascending=False).reset_index(drop=True)
display(cmp_df)

BEST_NAME = cmp_df.loc[0, "strategy"]; BEST_KW = candidates[BEST_NAME]
print("chosen threshold strategy:", BEST_NAME)

In [ ]:
# Final thresholds: fit on the full validation set using the chosen strategy.
THR = fit_thresholds(Y_val, val_probs, **BEST_KW)

fb_rows = []
for fb in ("ratio", "prob", "neutral"):
    P = apply_rule(val_probs, THR, fb)
    fb_rows.append({"fallback": fb, "val macro-F1": round(macro_f1(Y_val, P), 4), "val micro-F1": round(micro_f1(Y_val, P), 4)})
empty_share = (~(val_probs >= THR[None, :]).any(1)).mean()
print(f"Share of validation comments where the thresholds alone select nothing: {empty_share:.1%}")
display(pd.DataFrame(fb_rows))

FALLBACK = max(fb_rows, key=lambda r: r["val macro-F1"])["fallback"]
print("chosen fallback:", FALLBACK)

In [ ]:
# Summary on validation. Note: these are in-sample for the thresholds, so treat them as an optimistic upper bound;
# the cross-fitted number above is the honest estimate of what the rule should do on unseen data.
P_val = apply_rule(val_probs, THR, FALLBACK)
per_label = pd.DataFrame({
    "label": label_names, "val_support": Y_val.sum(0).astype(int),
    "threshold": THR.round(3), "F1": f1_per_label(Y_val, P_val).round(3),
}).sort_values("F1", ascending=False).reset_index(drop=True)

VAL_METRICS = {
    "macro_f1_at_0.5": macro_f1(Y_val, val_probs >= 0.5),
    "macro_f1_tuned_in_sample": macro_f1(Y_val, P_val),
    "micro_f1_tuned_in_sample": micro_f1(Y_val, P_val),
    "macro_f1_crossfit_heldout": float(cmp_df.loc[0, "held-out macro-F1 (mean)"]),
    "avg_labels_predicted": float(P_val.sum(1).mean()), "avg_labels_true": float(Y_val.sum(1).mean()),
}
print(json.dumps({k: round(v, 4) for k, v in VAL_METRICS.items()}, indent=2))
display(per_label)

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(10, 3.6))
ax.bar(per_label.label, per_label.F1); ax.set_xticklabels(per_label.label, rotation=75, ha="right")
ax.set_ylabel("validation F1"); ax.set_title("Per-label F1 (validation)"); plt.tight_layout(); plt.show()

**How to read the result.** Labels such as *gratitude*, *amusement* and *love* tend to be easy (their words are distinctive), while *relief*, *grief*, *realization* and *disappointment* are hard (they are rare, ambiguous, or both). That spread is normal for GoEmotions, and it is the reason Macro-F1 scores are much lower than micro-F1 scores. The honest number to quote for expected performance is the cross-fitted held-out Macro-F1, not the in-sample one.

## Part 5 · Saving the checkpoint

The folder `OUT_DIR` already contains the model weights (`model.safetensors`), `config.json` (which includes the label mapping) and the tokenizer files, saved during training. Now we add:

- `prediction_rule.json`: the thresholds, the fallback rule and the max sequence length,
- `label_mapping.json`: the original GoEmotions ID to name mapping,
- `run_config.json`: base model identifier **and revision**, dataset revision, seed, full configuration, parameter counts, library versions, validation metrics,
- `README_CHECKPOINT.txt`: states which checkpoint is the final one.

Because we fully fine-tuned the encoder, the saved weights are the *complete* model; no adapters or separate base-model download are needed. We still record the base model identifier and revision for provenance.

Then we zip the folder into a single archive.

In [ ]:
json.dump({
    "strategy": BEST_NAME, "strategy_params": BEST_KW, "fallback": FALLBACK,
    "max_len": CFG["max_len"], "threshold_grid": [float(GRID[0]), float(GRID[-1])],
    "thresholds": {n: float(t) for n, t in zip(label_names, THR)},
}, open(OUT_DIR / "prediction_rule.json", "w"), indent=2)

json.dump({str(i): n for i, n in enumerate(label_names)}, open(OUT_DIR / "label_mapping.json", "w"), indent=2)

json.dump({
    "config": CFG, "param_counts": PARAM_COUNTS, "versions": VERSIONS,
    "best_epoch": best["epoch"], "validation_metrics": VAL_METRICS,
}, open(OUT_DIR / "run_config.json", "w"), indent=2)

(OUT_DIR / "README_CHECKPOINT.txt").write_text(
f'''FINAL CHECKPOINT TO BE EVALUATED: this folder ({OUT_DIR.name}).
Base model: {CFG["model_name"]} @ revision {CFG["model_revision"]} (full fine-tuning; weights in model.safetensors are complete).
Load with AutoModelForSequenceClassification.from_pretrained(<this folder>) and AutoTokenizer.from_pretrained(<this folder>).
Prediction rule: per-label sigmoid thresholds + fallback, stored in prediction_rule.json.
The notebook's Part 6 (inference section) loads this folder and exposes predict(texts).
''')

zip_base = WORK / f"goemotions_checkpoint_{MODE}"
zip_path = shutil.make_archive(str(zip_base), "zip", root_dir=str(OUT_DIR.parent), base_dir=OUT_DIR.name)
print("archive:", zip_path, f"({os.path.getsize(zip_path) / 1e9:.2f} GB)")
print(sorted(p.name for p in OUT_DIR.iterdir()))

---
## Part 6 · Inference section (self-contained)

**This section does not depend on anything above.** To use it in a fresh session: start a new notebook session (or restart), skip Parts 0-5 entirely, and run only the cells below. It needs the checkpoint zip (or the unzipped folder) to be available, e.g. attached as a Kaggle dataset under `/kaggle/input/...` or sitting in the working directory. Set `CKPT_PATH` manually if auto-detection does not find it.

It defines `predict(texts)`, which takes a list of strings and returns, for each input and in the same order, a list of unique label names (at least one per input).

In [ ]:
# ===== INFERENCE: everything needed is imported/defined here =====
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"          # one GPU
import json, glob, zipfile
from pathlib import Path
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

CKPT_PATH = None    # <- optionally set to the .zip or the unzipped folder

def _find_checkpoint():
    if CKPT_PATH: return Path(CKPT_PATH)
    roots = ["/kaggle/working", "/kaggle/input", "."]
    for r in roots:                                           # unzipped folder first
        hits = glob.glob(os.path.join(r, "**", "prediction_rule.json"), recursive=True)
        if hits: return Path(hits[0]).parent
    for r in roots:                                           # then a zip
        hits = glob.glob(os.path.join(r, "**", "goemotions_checkpoint*.zip"), recursive=True)
        if hits: return Path(hits[0])
    raise FileNotFoundError("Checkpoint not found; set CKPT_PATH.")

def _resolve_dir(p):
    p = Path(p)
    if p.suffix == ".zip":
        dest = Path("/tmp/goemotions_ckpt"); dest.mkdir(exist_ok=True)
        with zipfile.ZipFile(p) as z: z.extractall(dest)
        p = Path(glob.glob(str(dest / "**" / "prediction_rule.json"), recursive=True)[0]).parent
    return p

CKPT_DIR = _resolve_dir(_find_checkpoint())
print("loading checkpoint from:", CKPT_DIR)

_dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
_tok = AutoTokenizer.from_pretrained(CKPT_DIR)
_model = AutoModelForSequenceClassification.from_pretrained(CKPT_DIR).to(_dev).eval()
_rule = json.load(open(CKPT_DIR / "prediction_rule.json"))
_labels = [_model.config.id2label[i] for i in range(_model.config.num_labels)]
_thr = np.array([_rule["thresholds"][n] for n in _labels], dtype=np.float32)
_MAX_LEN, _FALLBACK = _rule["max_len"], _rule["fallback"]
_NEUTRAL = _labels.index("neutral")

@torch.inference_mode()
def _probs(texts, batch_size=64):
    ids = _tok(list(texts), truncation=True, max_length=_MAX_LEN)["input_ids"]
    order = np.argsort([len(x) for x in ids])
    out = np.zeros((len(ids), len(_labels)), dtype=np.float32)
    for s in range(0, len(order), batch_size):
        idx = order[s:s + batch_size]
        batch = _tok.pad({"input_ids": [ids[i] for i in idx]}, return_tensors="pt").to(_dev)
        with torch.autocast(_dev.type, dtype=torch.float16, enabled=(_dev.type == "cuda")):
            logits = _model(**batch).logits
        out[idx] = torch.sigmoid(logits.float()).cpu().numpy()
    return out

def _decide(probs):
    pred = probs >= _thr[None, :]
    empty = np.flatnonzero(~pred.any(1))
    if len(empty):
        if _FALLBACK == "ratio":     idx = (probs[empty] / _thr[None, :]).argmax(1)
        elif _FALLBACK == "prob":    idx = probs[empty].argmax(1)
        else:                        idx = np.full(len(empty), _NEUTRAL)
        pred[empty, idx] = True
    return pred

def predict(texts):
    # texts: list[str]  ->  list[list[str]] (unique label names per input, same order as the input)
    texts = list(texts)
    if not texts: return []
    probs = _probs(texts); pred = _decide(probs); result = []
    for p, row in zip(probs, pred):
        idx = np.flatnonzero(row); idx = idx[np.argsort(-p[idx])]      # most confident first (order is not scored)
        result.append(list(dict.fromkeys(_labels[i] for i in idx)))
    return result

demo = ["Thank you so much, this really made my day!",
        "I can't believe he did that, I'm furious.",
        "Not sure what to think about this.",
        "I miss her so much it hurts."]
for t, r in zip(demo, predict(demo)):
    print(f"{r}  <-  {t}")

### Optional sanity check: does the reloaded checkpoint reproduce the validation score?

This reloads the *validation* split (not the test split) and compares the Macro-F1 from the reloaded checkpoint against the number recorded at training time. They should match closely (tiny differences can come from GPU non-determinism).

In [ ]:
from datasets import load_dataset
_run = json.load(open(CKPT_DIR / "run_config.json"))
_val = load_dataset(_run["config"]["dataset_name"], _run["config"]["dataset_config"],
                    revision=_run["config"]["dataset_revision"], split="validation")
_Y = np.zeros((len(_val), len(_labels)), dtype=bool)
for i, labs in enumerate(_val["labels"]): _Y[i, labs] = True

_P = _decide(_probs(_val["text"])); _tp = (_Y & _P).sum(0); _fp = (~_Y & _P).sum(0); _fn = (_Y & ~_P).sum(0)
_den = 2 * _tp + _fp + _fn
reloaded = float(np.where(_den > 0, 2 * _tp / np.maximum(_den, 1), 0).mean())
print(f"reloaded checkpoint, validation macro-F1: {reloaded:.4f}")
print(f"recorded at training time (in-sample):    {_run['validation_metrics']['macro_f1_tuned_in_sample']:.4f}")

---
## Part 7 · One-time test evaluation (off by default)

Everything is frozen at this point: checkpoint, thresholds, fallback. Flip the flag to `True` and run this cell **once**, at the very end, purely to report a number. Nothing is changed afterwards based on it.

In [ ]:
RUN_TEST_EVAL = False

if RUN_TEST_EVAL:
    _test = load_dataset(_run["config"]["dataset_name"], _run["config"]["dataset_config"],
                         revision=_run["config"]["dataset_revision"], split="test")
    _Yt = np.zeros((len(_test), len(_labels)), dtype=bool)
    for i, labs in enumerate(_test["labels"]): _Yt[i, labs] = True
    _Pt = np.array([[l in set(r) for l in _labels] for r in predict(_test["text"])])
    tp = (_Yt & _Pt).sum(0); fp = (~_Yt & _Pt).sum(0); fn = (_Yt & ~_Pt).sum(0)
    f1 = np.where((2*tp + fp + fn) > 0, 2*tp / np.maximum(2*tp + fp + fn, 1), 0)
    print(f"TEST macro-F1: {f1.mean():.4f}")
    print(f"TEST micro-F1: {2*tp.sum() / (2*tp.sum() + fp.sum() + fn.sum()):.4f}")
else:
    print("Test evaluation skipped (RUN_TEST_EVAL = False).")